# Projeto: Inserção dos dados na camada Gold - Previsão do Clima

### Descrição Geral
Este notebook documenta o processo de criação e atualização da tabela **`previsao_clima`** na camada **gold**, a partir da consolidação de dados enriquecidos da camada **gold**.
O objetivo principal é disponibilizar um conjunto de dados confiável e analiticamente robusto com as **previsões climáticas mais recentes**, para apoiar análises preditivas, geração de alertas e tomada de decisão em contextos sensíveis a variáveis meteorológicas.

### Informações Técnicas

- **Fonte de Dados**:
  * Tabelas da base Silver referentes a previsões climáticas:
    * `silver_previsao_clima`
    * `silver_chuvas_criticas`
  * Dados enriquecidos e tratados para a camada Gold

- **Tecnologias Utilizadas**:  
  - Apache Spark (PySpark)  
  - Databricks Notebooks
  - Catalog

- **Dados persistidos**:  
  - Data de carga (`dt_carga`)  
  - Data da previsão (`dt_previsao`)  
  - Nome do município (`nom_municipio`)  
  - Nome do estado (`nom_estado`)  
  - Temperatura máxima prevista (`temp_max`)  
  - Temperatura mínima prevista (`temp_min`)  
  - Umidade máxima prevista (`umid_max`)  
  - Umidade mínima prevista (`umid_min`)  
  - Precipitação prevista (mm) (`precipitacao`)  
  - Probabilidade de precipitação (%) (`prob_precipitacao`)  
  - Velocidade do vento (km/h) (`vel_vento`)  
  - Fonte da previsão do tempo (`flg_fonte_previsao`)  

---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                      |
|------------|-----------------|--------|------------------------------------------------|
| 21/05/2025 | Lucas Berta  | 1.0    | Criação da tabela previsão clima e persistência dos dados |
| 26/05/2025 | Evelise Dias      |  1.1  | 	Atualização da descrição geral do projeto e alinhamento com a arquitetura atual|

In [0]:
%run ../../src/utils/utils_run_gold

In [0]:
%run ./queries_gold

In [0]:
df = ler_tabelas(caminho_tabela_clima_gold)

In [0]:
window_spec = Window.partitionBy("flg_fonte_previsao").orderBy(col("dt_carga").desc())

In [0]:
df = df.withColumn("max_dt_carga", max("dt_carga").over(window_spec))

In [0]:
df = df.filter(col("dt_carga") == col("max_dt_carga")).drop("max_dt_carga")

In [0]:
df = df.filter(col("dt_previsao") >= current_date())

In [0]:
spark.sql(create_table_gold_previsao_clima)

In [0]:
df.write.mode("overwrite").saveAsTable(caminho_tabela_previsao_clima_gold)

In [0]:
%sql
select * from grupo01.gold.previsao_clima